In [ ]:
import os
import csv
import requests
import concurrent.futures

from tqdm import tqdm

/Users/jam/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [13]:
# Load Gutenberg metadata as a dictionary
metadata_dict = {}
with open('gutenberg_metadata.csv', mode='r', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    # Let's check the header fields to see what key to use
    print("CSV header fields:", reader.fieldnames)
    # We'll use 'book_number' as the unique key instead of 'id'
    for row in reader:
        key = row.get('book_number')
        if key is not None:
            metadata_dict[key] = row
print("metadata_dict is a dictionary with", len(metadata_dict), "entries.")

CSV header fields: ['book_number', 'title', 'author', 'language', 'genres', 'plain_text_url']
metadata_dict is a dictionary with 24278 entries.


In [14]:
def download_single_book(args):
    book_id, book_info = args
    url = book_info.get('plain_text_url', '').strip()
    if not url:
        print(f"Book {book_id}: No plain_text_url found, skipping.")
        return

    output_dir = "project_books_raw"
    title = book_info.get('title', f"book_{book_id}")
    safe_title = "".join(c for c in title if c.isalnum() or c in (' ', '-', '_')).rstrip()
    filename = f"{book_id}_{safe_title}.txt"
    filepath = os.path.join(output_dir, filename)

    # Skip if already downloaded
    if os.path.exists(filepath):
        return

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(response.text)
    except Exception as e:
        print(f"Failed to download {title} ({url}): {e}")

# Ensure the output directory exists before threading
output_dir = "project_books_raw"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [15]:
# Download books using threads (10 workers)
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    list(tqdm(executor.map(download_single_book, metadata_dict.items()), total=len(metadata_dict), desc="Downloading books (threaded)"))

Failed to download Glinda of OzIn Which Are Related the Exciting Experiences of Princess Ozma of Oz, and Dorothy, in Their Hazardous Journey to the Home of the Flatheads, and to the Magic Isle of the Skeezers, and How They Were Rescued from Dire Peril by the Sorcery of Glinda the Good (https://www.gutenberg.org/ebooks/961.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/961_Glinda of OzIn Which Are Related the Exciting Experiences of Princess Ozma of Oz and Dorothy in Their Hazardous Journey to the Home of the Flatheads and to the Magic Isle of the Skeezers and How They Were Rescued from Dire Peril by the Sorcery of Glinda the Good.txt'


Failed to download The Marvelous Exploits of Paul BunyanAs Told in the Camps of the White Pine Lumbermen for Generations During Which Time the Loggers Have Pioneered the Way Through the North Woods From Maine to California. Collected from Various Sources and Embellished for Publication (https://www.gutenberg.org/ebooks/5800.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/5800_The Marvelous Exploits of Paul BunyanAs Told in the Camps of the White Pine Lumbermen for Generations During Which Time the Loggers Have Pioneered the Way Through the North Woods From Maine to California Collected from Various Sources and Embellished for Publication.txt'


Failed to download Mary Schweidler, the amber witchThe most interesting trial for witchcraft ever known, printed from an imperfect manuscript by her father, Abraham Schweidler, the pastor of Coserow in the island of Usedom / edited by W. Meinhold ; translated from the German by Lady Duff Gordon. (https://www.gutenberg.org/ebooks/8743.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/8743_Mary Schweidler the amber witchThe most interesting trial for witchcraft ever known printed from an imperfect manuscript by her father Abraham Schweidler the pastor of Coserow in the island of Usedom  edited by W Meinhold  translated from the German by Lady Duff Gordon.txt'


Failed to download The Boats of the "Glen Carrig"Being an account of their Adventures in the Strange places of the Earth, after the foundering of the good ship Glen Carrig through striking upon a hidden rock in the unknown seas to the Southward; as told by John Winterstraw, Gent., to his son James Winterstraw, in the year 1757, and by him committed very properly and legibly to manuscript (https://www.gutenberg.org/ebooks/10542.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/10542_The Boats of the Glen CarrigBeing an account of their Adventures in the Strange places of the Earth after the foundering of the good ship Glen Carrig through striking upon a hidden rock in the unknown seas to the Southward as told by John Winterstraw Gent to his son James Winterstraw in the year 1757 and by him committed very properly and legibly to manuscript.txt'


Failed to download The Fortunate FoundlingsBeing the Genuine History of Colonel M——Rs, and His Sister, Madam Du P——Y, the Issue of the Hon. Ch——Es M——Rs, Son of the Late Duke of R—— L——D. Containing Many Wonderful Accidents That Befel Them in Their Travels, and Interspersed with the Characters and Adventures of Several Persons of Condition, In the Most Polite Courts of Europe. the Whole Calculated for the Entertainment and Improvement of the Youth of Both Sexes. (https://www.gutenberg.org/ebooks/10804.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/10804_The Fortunate FoundlingsBeing the Genuine History of Colonel MRs and His Sister Madam Du PY the Issue of the Hon ChEs MRs Son of the Late Duke of R LD Containing Many Wonderful Accidents That Befel Them in Their Travels and Interspersed with the Characters and Adventures of Several Persons of Condition In the Most Polite Courts of Europe the Whole Calculated for the Entertainment and Improvement of the Youth of Both Sexes

...

Failed to download Lost with Lieutenant PikeHow from the Pawnee Village the boy named Scar Head marched with the young American Chief clear into the Snowy Mountains; how in the dead of winter they searched for the Lost River and thought that they had found it; and how the Spanish Soldiery came upon them and took them down to Santa Fé of New Mexico, where another surprise awaited them (https://www.gutenberg.org/ebooks/67142.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/67142_Lost with Lieutenant PikeHow from the Pawnee Village the boy named Scar Head marched with the young American Chief clear into the Snowy Mountains how in the dead of winter they searched for the Lost River and thought that they had found it and how the Spanish Soldiery came upon them and took them down to Santa Fé of New Mexico where another surprise awaited them.txt'


Failed to download Library of the best American literatureContaining the lives of our authors in story form, their portraits, their homes, and their personal traits, how they worked and what they wrote; choice selections from eminent writers, embracing great American poets and novelists, foremost women in American letters, distinguished critics and essayists, our national humorists, noted journalists and magazine contributors, popular writers for young people, great orators and public lecturers (https://www.gutenberg.org/ebooks/69620.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/69620_Library of the best American literatureContaining the lives of our authors in story form their portraits their homes and their personal traits how they worked and what they wrote choice selections from eminent writers embracing great American poets and novelists foremost women in American letters distinguished critics and essayists our national humorists noted journalists and magazine cont

Failed to download These charming people :  being a tapestry of the fortunes, follies, adventures, gallantries and general activities of Shelmerdene (that lovely lady), Lord Tarlyon, Mr. Michael Wagstaffe, Mr. Ralph Wyndham Trevor and some others of their friends of the lighter sort (https://www.gutenberg.org/ebooks/72477.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/72477_These charming people   being a tapestry of the fortunes follies adventures gallantries and general activities of Shelmerdene that lovely lady Lord Tarlyon Mr Michael Wagstaffe Mr Ralph Wyndham Trevor and some others of their friends of the lighter sort.txt'


Failed to download May Fair :  being an entertainment purporting to reveal to gentlefolk the real state of affairs existing in the very heart of London during the fifteenth and sixteenth years of the reign of His Majesty King George the Fifth: together with suitable reflections on the last follies, misadventures and galanteries of these charming people (https://www.gutenberg.org/ebooks/72651.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/72651_May Fair   being an entertainment purporting to reveal to gentlefolk the real state of affairs existing in the very heart of London during the fifteenth and sixteenth years of the reign of His Majesty King George the Fifth together with suitable reflections on the last follies misadventures and galanteries of these charming people.txt'


Failed to download Next year :  a semi-historical account of the exploits and exploitations of the far-famed Barr Colonists, who, led by an unscrupulous Church of England parson, adventured deep into the wilderness of Canada's great North-West in the early days of the twentieth century (https://www.gutenberg.org/ebooks/73709.txt.utf-8): [Errno 63] File name too long: 'project_books_raw/73709_Next year   a semi-historical account of the exploits and exploitations of the far-famed Barr Colonists who led by an unscrupulous Church of England parson adventured deep into the wilderness of Canadas great North-West in the early days of the twentieth century.txt'
